## WVS Wave 7 – Variable Inventory and First Data Quality Exploration
===================================================================

### Purpose:
    1. Load the WVS CSV file.
    2. Create an inventory of all columns.
    3. Summarise data types, missing values, unique values and examples.
    4. Flag possible special numeric codes for review (not automatically missing).
    5. Create a preliminary, name-based variable classification.
    6. Export results to Excel and CSV for presentation and further EDA.

### Important:
    - The classification is heuristic and must be checked against the WVS codebook.
    - Negative values are flagged, not automatically replaced with NaN.
    - Numeric storage type does not tell us whether a variable is continuous,
      ordinal, categorical or binary.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# 1. File paths and settings
# ---------------------------------------------------------------------

# Put this Python file in the same folder as the WVS CSV file,
# or change DATA_FILE to the full path of your CSV.
BASE_DIR = Path(__file__).resolve().parent
DATA_FILE = BASE_DIR / "WVS_Cross-National_Wave_7_csv_v6_0.csv"
OUTPUT_DIR = BASE_DIR / "WVS_analysis_output"

# Common negative codes are flagged for inspection only.
# Their meanings must be confirmed in the official codebook.
POSSIBLE_SPECIAL_CODES = [-1, -2, -3, -4, -5, -9, -99, -999]


# ---------------------------------------------------------------------
# 2. Preliminary variable classification
# ---------------------------------------------------------------------

def classify_variable(column_name: str) -> str:
    """
    Assign a preliminary category based on the column name.
    This is only a first-pass grouping, not an official WVS classification.
    """
    name = str(column_name).strip()
    upper = name.upper()

    metadata_patterns = [
        r"^A_", r"^B_", r"^S_", r"^G_", r"^N_", r"^W_",
        r"WEIGHT", r"^ID", r"^COUNTRY", r"^YEAR", r"^INTERVIEW",
        r"^MODE", r"^REGION", r"^PSU", r"^STRAT"
    ]
    if any(re.search(pattern, upper) for pattern in metadata_patterns):
        return "Survey metadata / sampling / respondent context"

    external_patterns = [
        "GDP", "GINI", "UNEMPLOY", "LIFEEXPECT", "HDI", "INTERNET",
        "V2X", "CORRUPTION", "INFLATION", "POVERTY", "FREEDOM",
        "ELECTRICITY", "URBAN", "POPULATION", "GOVERNANCE"
    ]
    if any(term in upper for term in external_patterns):
        return "External country-level indicators"

    derived_patterns = [
        "INDEX", "INDICES", "SCALE", "FACTOR", "DIMENSION",
        "TRUST", "HAPPINESS", "WELLBEING", "WELL_BEING"
    ]
    if any(term in upper for term in derived_patterns) and not re.match(r"^Q\d+", upper):
        return "Possible derived index / scale"

    if re.match(r"^Q\d+", upper):
        return "Survey question / question-derived field"

    if any(term in upper for term in ["PARTY", "GPS", "LAT", "LON", "LONGITUDE", "LATITUDE"]):
        return "Other / requires codebook review"

    return "Other / requires codebook review"


# ---------------------------------------------------------------------
# 3. Helper functions
# ---------------------------------------------------------------------

def example_values(series: pd.Series, limit: int = 8) -> str:
    """Return a compact string of example non-missing values."""
    values = series.dropna().drop_duplicates().head(limit).tolist()
    return " | ".join(str(value) for value in values)


def possible_special_code_summary(series: pd.Series) -> tuple[int, str]:
    """Count listed candidate codes; do not assume they are all missing."""
    counts = series.value_counts(dropna=True)
    found = {
        code: int(counts.get(code, 0))
        for code in POSSIBLE_SPECIAL_CODES
        if counts.get(code, 0) > 0
    }
    total = sum(found.values())
    detail = "; ".join(f"{code}: {count}" for code, count in found.items())
    return total, detail


# ---------------------------------------------------------------------
# 4. Main analysis
# ---------------------------------------------------------------------

def main():
    if not DATA_FILE.exists():
        print("\nERROR: CSV file was not found.")
        print(f"Expected location: {DATA_FILE}")
        print("\nMove this script into the same folder as the CSV,")
        print("or edit DATA_FILE near the top of this script.")
        sys.exit(1)

    OUTPUT_DIR.mkdir(exist_ok=True)

    print("=" * 70)
    print("WVS WAVE 7 – VARIABLE INVENTORY")
    print("=" * 70)
    print(f"\nReading file: {DATA_FILE.name}")
    print("This may take a little while because the source file is large...")

    # Read the complete dataset to inspect all columns.
    df = pd.read_csv(DATA_FILE, low_memory=False)

    n_rows, n_columns = df.shape
    print("\nDATASET OVERVIEW")
    print("-" * 70)
    print(f"Rows (respondents): {n_rows:,}")
    print(f"Columns (variables): {n_columns:,}")

    # Look for likely country identifiers.
    country_col = next(
        (col for col in ["B_COUNTRY_ALPHA", "COUNTRY_ALPHA", "B_COUNTRY"] if col in df.columns),
        None
    )
    if country_col:
        print(f"Country/region column: {country_col}")
        print(f"Distinct country/region values: {df[country_col].nunique(dropna=True):,}")
    else:
        print("Country/region column not identified automatically.")

    # Data type counts.
    dtype_counts = df.dtypes.astype(str).value_counts()
    print("\nDATA TYPES")
    print("-" * 70)
    print(dtype_counts.to_string())

    # Build one inventory row per column.
    inventory_rows = []
    for index, column in enumerate(df.columns, start=1):
        series = df[column]
        candidate_count, candidate_detail = possible_special_code_summary(series)

        inventory_rows.append({
            "Column number": index,
            "Variable": column,
            "Data type": str(series.dtype),
            "Missing count (NaN)": int(series.isna().sum()),
            "Missing percentage (NaN)": round(series.isna().mean() * 100, 2),
            "Unique values (excluding NaN)": int(series.nunique(dropna=True)),
            "Candidate special-code count": candidate_count,
            "Candidate special-code details": candidate_detail,
            "Example values": example_values(series),
            "Preliminary category": classify_variable(column)
        })

        if index % 50 == 0:
            print(f"Processed {index}/{n_columns} columns...")

    inventory = pd.DataFrame(inventory_rows)

    # Category summary.
    category_summary = (
        inventory.groupby("Preliminary category", dropna=False)
        .agg(
            Number_of_variables=("Variable", "count"),
            Mean_missing_percentage=("Missing percentage (NaN)", "mean")
        )
        .reset_index()
        .sort_values("Number_of_variables", ascending=False)
    )
    category_summary["Mean_missing_percentage"] = (
        category_summary["Mean_missing_percentage"].round(2)
    )

    # Highest missingness.
    top_missing = inventory.sort_values(
        "Missing percentage (NaN)", ascending=False
    ).head(30)

    # Candidate special-code summary.
    special_codes = inventory[
        inventory["Candidate special-code count"] > 0
    ].sort_values("Candidate special-code count", ascending=False)

    # Save CSV inventory.
    inventory_csv = OUTPUT_DIR / "WVS_variable_inventory.csv"
    inventory.to_csv(inventory_csv, index=False, encoding="utf-8-sig")

    # Save Excel workbook with multiple sheets.
    excel_file = OUTPUT_DIR / "WVS_variable_inventory_and_quality.xlsx"
    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        inventory.to_excel(writer, sheet_name="Variable inventory", index=False)
        category_summary.to_excel(writer, sheet_name="Category summary", index=False)
        top_missing.to_excel(writer, sheet_name="Top missingness", index=False)
        special_codes.to_excel(writer, sheet_name="Candidate special codes", index=False)

        overview = pd.DataFrame({
            "Measure": [
                "Rows",
                "Columns",
                "Distinct countries/regions" if country_col else "Country column found",
                "Numeric columns",
                "Non-numeric columns"
            ],
            "Value": [
                n_rows,
                n_columns,
                df[country_col].nunique(dropna=True) if country_col else "Not identified",
                int(df.select_dtypes(include=np.number).shape[1]),
                int(df.select_dtypes(exclude=np.number).shape[1])
            ]
        })
        overview.to_excel(writer, sheet_name="Dataset overview", index=False)

    # Print concise results for the VS Code terminal.
    print("\nPRELIMINARY CATEGORY SUMMARY")
    print("-" * 70)
    print(category_summary.to_string(index=False))

    print("\nTOP 15 VARIABLES BY NaN MISSINGNESS")
    print("-" * 70)
    print(
        top_missing[["Variable", "Missing percentage (NaN)", "Unique values (excluding NaN)"]]
        .head(15)
        .to_string(index=False)
    )

    print("\nFILES CREATED")
    print("-" * 70)
    print(f"Excel report: {excel_file}")
    print(f"CSV inventory: {inventory_csv}")
    print("\nAnalysis completed.")
    print("Remember: candidate special codes and preliminary categories need codebook review.")


if __name__ == "__main__":
    main()
